# Feature Selection - Measuring Feature Redundancy

This notebook implements correlation-based, distance-based, and binary-similarity methods for all 16 questions in the assignment.

In [1]:
import numpy as np
import pandas as pd
from scipy.spatial.distance import jaccard, hamming

## Question 1: Correlation-Based Feature Redundancy

A correlation matrix shows linear relationships. A pair with correlation greater than 0.90 may contain redundant information.

In [2]:
data = {
    'F1': [10, 20, 30, 40, 50],
    'F2': [12, 24, 36, 48, 60],
    'F3': [50, 40, 30, 20, 10],
    'F4': [15, 25, 20, 35, 30]
}
df = pd.DataFrame(data)

correlation_matrix = df.corr()
print("Correlation matrix:")
print(correlation_matrix)

redundant_pairs = []
for i, column_1 in enumerate(correlation_matrix.columns):
    for column_2 in correlation_matrix.columns[i + 1:]:
        correlation = correlation_matrix.loc[column_1, column_2]
        if correlation > 0.90:
            redundant_pairs.append((column_1, column_2, correlation))

print("\nFeature pairs with positive correlation greater than 0.90:")
for pair in redundant_pairs:
    print(pair[0], "-", pair[1], ":", round(pair[2], 4))

Correlation matrix:
     F1   F2   F3   F4
F1  1.0  1.0 -1.0  0.8
F2  1.0  1.0 -1.0  0.8
F3 -1.0 -1.0  1.0 -0.8
F4  0.8  0.8 -0.8  1.0

Feature pairs with positive correlation greater than 0.90:
F1 - F2 : 1.0


**Observation:** F1 and F2 have a strong positive correlation and may provide overlapping information.

## Question 2: Distance-Based Feature Similarity

Euclidean distance measures straight-line distance, Manhattan distance sums absolute coordinate differences, and cosine similarity measures direction.

In [3]:
F1 = np.array([2, 4, 6, 8, 10])
F2 = np.array([3, 5, 7, 9, 11])

euclidean_distance = np.linalg.norm(F1 - F2)
manhattan_distance = np.sum(np.abs(F1 - F2))
cosine_similarity = np.dot(F1, F2) / (np.linalg.norm(F1) * np.linalg.norm(F2))

print("Euclidean distance:", euclidean_distance)
print("Manhattan distance:", manhattan_distance)
print("Cosine similarity:", cosine_similarity)

Euclidean distance: 2.23606797749979
Manhattan distance: 5
Cosine similarity: 0.9984038297885897


**Observation:** The vectors differ by 1 in every position, while cosine similarity indicates that they point in very similar directions.

## Question 3: Comparing Feature Vectors Using Different Distance Metrics

The L2 norm calculates Euclidean distance and the L1 norm calculates Manhattan distance.

In [4]:
F1 = np.array([1, 3, 5, 7])
F2 = np.array([2, 3, 6, 8])

euclidean_distance = np.linalg.norm(F1 - F2, ord=2)
manhattan_distance = np.linalg.norm(F1 - F2, ord=1)

print("Euclidean (L2) distance:", euclidean_distance)
print("Manhattan (L1) distance:", manhattan_distance)
print("Smaller distance:", "Euclidean" if euclidean_distance < manhattan_distance else "Manhattan")

Euclidean (L2) distance: 1.7320508075688772
Manhattan (L1) distance: 3.0
Smaller distance: Euclidean


**Observation:** Manhattan distance is larger because it sums all absolute differences.

## Question 4: Measuring Redundancy Between Binary Features

Binary feature counts are n11, n00, n10, and n01. Jaccard ignores n00, while SMC includes both matches.

In [5]:
F1 = np.array([1, 1, 0, 1, 0, 0, 1, 0])
F2 = np.array([1, 0, 0, 1, 1, 0, 1, 0])

hamming_distance = np.sum(F1 != F2)
n11 = np.sum((F1 == 1) & (F2 == 1))
n00 = np.sum((F1 == 0) & (F2 == 0))
n10 = np.sum((F1 == 1) & (F2 == 0))
n01 = np.sum((F1 == 0) & (F2 == 1))
jaccard_similarity = n11 / (n11 + n10 + n01)
jaccard_distance = 1 - jaccard_similarity
smc = (n11 + n00) / len(F1)

print("Hamming distance:", hamming_distance)
print("n11:", n11, "n00:", n00, "n10:", n10, "n01:", n01)
print("Jaccard similarity:", jaccard_similarity)
print("Jaccard distance:", jaccard_distance)
print("Simple Matching Coefficient:", smc)

Hamming distance: 2
n11: 3 n00: 3 n10: 1 n01: 1
Jaccard similarity: 0.6
Jaccard distance: 0.4
Simple Matching Coefficient: 0.75


**Observation:** The features have 2 differing positions and a Jaccard similarity of 0.75.

## Question 5: Comparing Binary Similarity Measures

Hamming distance counts mismatches, Jaccard distance focuses on 1/1 overlap, and SMC considers both types of matches.

In [6]:
F1 = np.array([1, 0, 1, 1, 0, 1, 0, 0])
F2 = np.array([1, 0, 0, 1, 1, 1, 0, 1])

hamming_distance = np.sum(F1 != F2)
n11 = np.sum((F1 == 1) & (F2 == 1))
n10 = np.sum((F1 == 1) & (F2 == 0))
n01 = np.sum((F1 == 0) & (F2 == 1))
n00 = np.sum((F1 == 0) & (F2 == 0))
jaccard_similarity = n11 / (n11 + n10 + n01)
jaccard_distance = 1 - jaccard_similarity
smc = (n11 + n00) / len(F1)

print("Hamming distance:", hamming_distance)
print("Jaccard distance:", jaccard_distance)
print("Simple Matching Coefficient:", smc)

Hamming distance: 3
Jaccard distance: 0.5
Simple Matching Coefficient: 0.625


**Observation:** SMC includes matching zeros and is therefore often more influenced by common absence than Jaccard distance.

## Question 6: Identifying Redundant Features

Correlations above 0.95 indicate strong positive linear redundancy. Removing one feature from each redundant pair reduces duplication.

In [7]:
data = {
    'Study_Hours': [2, 4, 6, 8, 10, 12],
    'Practice_Hours': [3, 6, 9, 12, 15, 18],
    'Sleep_Hours': [8, 7, 8, 6, 7, 8],
    'Screen_Time': [2, 3, 4, 5, 6, 7]
}
df = pd.DataFrame(data)

correlation_matrix = df.corr()
print("Correlation matrix:")
print(correlation_matrix)

redundant_pairs = []
for i, column_1 in enumerate(correlation_matrix.columns):
    for column_2 in correlation_matrix.columns[i + 1:]:
        correlation = correlation_matrix.loc[column_1, column_2]
        if correlation > 0.95:
            redundant_pairs.append((column_1, column_2))

print("\nRedundant feature pairs:")
for pair in redundant_pairs:
    print(pair[0], "-", pair[1])

features_to_remove = {pair[1] for pair in redundant_pairs}
reduced_df = df.drop(columns=list(features_to_remove))
print("\nDataFrame after removing redundant features:")
print(reduced_df)

Correlation matrix:
                Study_Hours  Practice_Hours  Sleep_Hours  Screen_Time
Study_Hours        1.000000        1.000000    -0.130931     1.000000
Practice_Hours     1.000000        1.000000    -0.130931     1.000000
Sleep_Hours       -0.130931       -0.130931     1.000000    -0.130931
Screen_Time        1.000000        1.000000    -0.130931     1.000000

Redundant feature pairs:
Study_Hours - Practice_Hours
Study_Hours - Screen_Time
Practice_Hours - Screen_Time

DataFrame after removing redundant features:
   Study_Hours  Sleep_Hours
0            2            8
1            4            7
2            6            8
3            8            6
4           10            7
5           12            8


**Observation:** Study_Hours and Practice_Hours are redundant; Practice_Hours is removed.

## Question 7: Correlation Analysis of Student Performance

Pearson correlation measures linear relationship strength. The pair with the larger positive coefficient has the stronger positive relationship.

In [8]:
data = {
    'Assignment': [40, 50, 60, 70, 80, 90],
    'Internal': [42, 48, 62, 68, 82, 88],
    'Attendance': [60, 65, 70, 75, 80, 85],
    'Project': [30, 45, 50, 65, 75, 85]
}
df = pd.DataFrame(data)

assignment_internal = df['Assignment'].corr(df['Internal'])
attendance_project = df['Attendance'].corr(df['Project'])
print("Assignment vs Internal correlation:", assignment_internal)
print("Attendance vs Project correlation:", attendance_project)

stronger_pair = ("Assignment and Internal" if assignment_internal > attendance_project
                 else "Attendance and Project")
print("Stronger positive correlation:", stronger_pair)

Assignment vs Internal correlation: 0.9933445827155672
Attendance vs Project correlation: 0.9950735794761252
Stronger positive correlation: Attendance and Project


**Observation:** The pair with the higher positive correlation is identified from the calculated coefficients.

## Question 8: Finding the Most Similar Feature Pair

Cosine similarity is calculated pairwise between rows. A value of 1 indicates identical direction.

In [9]:
data = np.array([
    [2, 4, 6, 8, 10],
    [3, 6, 9, 12, 15],
    [10, 8, 6, 4, 2],
    [4, 5, 7, 8, 9]
])

feature_names = ["F1", "F2", "F3", "F4"]
similarity_matrix = np.zeros((len(data), len(data)))

for i in range(len(data)):
    for j in range(len(data)):
        similarity_matrix[i, j] = np.dot(data[i], data[j]) / (
            np.linalg.norm(data[i]) * np.linalg.norm(data[j])
        )

similarity_df = pd.DataFrame(similarity_matrix, index=feature_names, columns=feature_names)
print("Cosine similarity matrix:")
print(similarity_df)

best_pair = None
highest_similarity = -2
for i in range(len(data)):
    for j in range(i + 1, len(data)):
        if similarity_matrix[i, j] > highest_similarity:
            highest_similarity = similarity_matrix[i, j]
            best_pair = (feature_names[i], feature_names[j])

print("\nHighest-similarity pair:", best_pair)
print("Cosine similarity:", highest_similarity)

Cosine similarity matrix:
          F1        F2        F3        F4
F1  1.000000  1.000000  0.636364  0.985151
F2  1.000000  1.000000  0.636364  0.985151
F3  0.636364  0.636364  1.000000  0.756455
F4  0.985151  0.985151  0.756455  1.000000

Highest-similarity pair: ('F1', 'F2')
Cosine similarity: 1.0


**Observation:** F1 and F2 have the highest cosine similarity because F2 is a scalar multiple of F1.

## Question 9: Applying a Correlation Threshold

Only absolute correlations above 0.85 are considered redundant. Iterating over the upper triangular matrix avoids duplicate pairs.

In [10]:
corr_data = {
    'F1': [1.00, 0.92, 0.35, 0.20],
    'F2': [0.92, 1.00, 0.40, 0.25],
    'F3': [0.35, 0.40, 1.00, 0.88],
    'F4': [0.20, 0.25, 0.88, 1.00]
}
corr_df = pd.DataFrame(corr_data, index=['F1', 'F2', 'F3', 'F4'])
threshold = 0.85

print("Correlation matrix:")
print(corr_df)
print("\nPairs with absolute correlation greater than", threshold, ":")
for i, row_name in enumerate(corr_df.index):
    for column_name in corr_df.columns[i + 1:]:
        correlation = corr_df.loc[row_name, column_name]
        if abs(correlation) > threshold:
            print(row_name, "-", column_name, ":", correlation)

Correlation matrix:
      F1    F2    F3    F4
F1  1.00  0.92  0.35  0.20
F2  0.92  1.00  0.40  0.25
F3  0.35  0.40  1.00  0.88
F4  0.20  0.25  0.88  1.00

Pairs with absolute correlation greater than 0.85 :
F1 - F2 : 0.92
F3 - F4 : 0.88


**Observation:** F1-F2 and F3-F4 exceed the threshold and are identified exactly once.

## Question 10: Effect of Distance on Feature Vectors

A Euclidean distance of zero means the vectors have identical values in every position.

In [11]:
F1 = np.array([2, 5, 8, 10])
F2 = np.array([4, 5, 6, 12])
F3 = np.array([2, 5, 8, 10])

distance_f1_f2 = np.linalg.norm(F1 - F2)
distance_f1_f3 = np.linalg.norm(F1 - F3)

print("Euclidean distance between F1 and F2:", distance_f1_f2)
print("Euclidean distance between F1 and F3:", distance_f1_f3)
print("Identical pair:", "F1 and F3" if distance_f1_f3 == 0 else "No identical pair")

Euclidean distance between F1 and F2: 3.4641016151377544
Euclidean distance between F1 and F3: 0.0
Identical pair: F1 and F3


**Observation:** F1 and F3 are identical, so their Euclidean distance is zero.

## Question 11: Binary Feature Comparison

SMC combines both 11 matches and 00 matches; Hamming distance counts differing positions.

In [12]:
F1 = np.array([1, 1, 0, 0, 1, 0, 1, 0])
F2 = np.array([1, 0, 0, 1, 1, 0, 0, 0])

n11 = np.sum((F1 == 1) & (F2 == 1))
n00 = np.sum((F1 == 0) & (F2 == 0))
hamming_distance = np.sum(F1 != F2)
smc = (n11 + n00) / len(F1)

print("n11:", n11)
print("n00:", n00)
print("Simple Matching Coefficient:", smc)
print("Hamming distance:", hamming_distance)

n11: 2
n00: 3
Simple Matching Coefficient: 0.625
Hamming distance: 3


**Observation:** Both matches and mismatches are explicitly counted.

## Question 12: Jaccard Similarity for Binary Features

Jaccard similarity is n11 divided by n11 + n10 + n01. It ignores positions where both values are zero.

In [13]:
Product_A = np.array([1, 1, 0, 1, 0, 1, 0, 0, 1])
Product_B = np.array([1, 0, 0, 1, 1, 1, 0, 1, 0])

n11 = np.sum((Product_A == 1) & (Product_B == 1))
n10 = np.sum((Product_A == 1) & (Product_B == 0))
n01 = np.sum((Product_A == 0) & (Product_B == 1))
jaccard_similarity = n11 / (n11 + n10 + n01)
jaccard_distance = 1 - jaccard_similarity

print("Number of common purchases (n11):", n11)
print("Jaccard similarity:", jaccard_similarity)
print("Jaccard distance:", jaccard_distance)
print("Interpretation:",
      "Highly similar" if jaccard_similarity >= 0.75 else "Moderately or weakly similar")

Number of common purchases (n11): 3
Jaccard similarity: 0.42857142857142855
Jaccard distance: 0.5714285714285714
Interpretation: Moderately or weakly similar


**Observation:** The interpretation is based on the calculated Jaccard similarity.

## Question 13: Comparing Multiple Binary Features

Hamming distance counts unequal positions. The pair with the smallest distance is the most similar.

In [14]:
data = {
    'F1': [1, 1, 0, 1, 0, 1],
    'F2': [1, 1, 0, 1, 0, 1],
    'F3': [0, 1, 1, 0, 1, 0],
    'F4': [1, 0, 1, 0, 1, 1]
}
df = pd.DataFrame(data)

hamming_f1_f2 = np.sum(df['F1'].to_numpy() != df['F2'].to_numpy())
hamming_f1_f3 = np.sum(df['F1'].to_numpy() != df['F3'].to_numpy())

print("Hamming distance F1-F2:", hamming_f1_f2)
print("Hamming distance F1-F3:", hamming_f1_f3)
print("Most similar pair:",
      "F1 and F2" if hamming_f1_f2 < hamming_f1_f3 else "F1 and F3")
print("A Hamming distance of zero means the two vectors are identical at every position.")

Hamming distance F1-F2: 0
Hamming distance F1-F3: 5
Most similar pair: F1 and F2
A Hamming distance of zero means the two vectors are identical at every position.


**Observation:** F1 and F2 have a Hamming distance of zero and are most similar.

## Question 14: Calculating Minkowski Distance

Minkowski distance uses parameter r. r = 1 is Manhattan distance and r = 2 is Euclidean distance.

In [15]:
F1 = np.array([2, 4, 6, 8])
F2 = np.array([5, 6, 8, 10])

minkowski_r1 = np.linalg.norm(F1 - F2, ord=1)
minkowski_r2 = np.linalg.norm(F1 - F2, ord=2)
minkowski_r3 = np.linalg.norm(F1 - F2, ord=3)

print("Minkowski distance r = 1 (Manhattan):", minkowski_r1)
print("Minkowski distance r = 2 (Euclidean):", minkowski_r2)
print("Minkowski distance r = 3:", minkowski_r3)

Minkowski distance r = 1 (Manhattan): 9.0
Minkowski distance r = 2 (Euclidean): 4.58257569495584
Minkowski distance r = 3: 3.708429769266189


**Observation:** r = 1 gives Manhattan distance, while r = 2 gives Euclidean distance.

## Question 15: Feature Similarity Using Cosine Similarity

Cosine similarity compares vector direction. A higher value indicates greater directional similarity.

In [16]:
F1 = np.array([1, 2, 3, 4])
F2 = np.array([2, 4, 6, 8])
F3 = np.array([4, 3, 2, 1])

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

cosine_f1_f2 = cosine_similarity(F1, F2)
cosine_f1_f3 = cosine_similarity(F1, F3)

print("Cosine similarity F1-F2:", cosine_f1_f2)
print("Cosine similarity F1-F3:", cosine_f1_f3)
print("Greater similarity:",
      "F1 and F2" if cosine_f1_f2 > cosine_f1_f3 else "F1 and F3")

Cosine similarity F1-F2: 1.0
Cosine similarity F1-F3: 0.6666666666666666
Greater similarity: F1 and F2


**Observation:** F1 and F2 have greater cosine similarity because F2 is a scaled version of F1.

## Question 16: Identifying Redundancy in a Dataset

An absolute correlation of 1 indicates a perfectly redundant pair. One feature from each pair can be removed.

In [17]:
data = {
    'Temperature': [20, 25, 30, 35, 40, 45],
    'Temp_Celsius': [20, 25, 30, 35, 40, 45],
    'Humidity': [60, 65, 55, 70, 50, 75],
    'Pressure': [1000, 1005, 1010, 1015, 1020, 1025]
}
df = pd.DataFrame(data)

correlation_matrix = df.corr()
print("Correlation matrix:")
print(correlation_matrix)

perfect_pairs = []
for i, column_1 in enumerate(correlation_matrix.columns):
    for column_2 in correlation_matrix.columns[i + 1:]:
        correlation = correlation_matrix.loc[column_1, column_2]
        if abs(correlation) == 1:
            perfect_pairs.append((column_1, column_2))

print("\nPerfectly correlated feature pairs:")
for pair in perfect_pairs:
    print(pair[0], "-", pair[1])

features_to_remove = {pair[1] for pair in perfect_pairs}
resulting_df = df.drop(columns=list(features_to_remove))
print("\nResulting DataFrame:")
print(resulting_df)

Correlation matrix:
              Temperature  Temp_Celsius  Humidity  Pressure
Temperature      1.000000      1.000000  0.257143  1.000000
Temp_Celsius     1.000000      1.000000  0.257143  1.000000
Humidity         0.257143      0.257143  1.000000  0.257143
Pressure         1.000000      1.000000  0.257143  1.000000

Perfectly correlated feature pairs:
Temperature - Temp_Celsius
Temperature - Pressure
Temp_Celsius - Pressure

Resulting DataFrame:
   Temperature  Humidity
0           20        60
1           25        65
2           30        55
3           35        70
4           40        50
5           45        75


**Observation:** Temperature and Temp_Celsius are perfectly correlated; Temp_Celsius is removed.